In [1]:
!pip install -q --no-cache-dir \
  transformers accelerate datasets huggingface_hub sentencepiece bitsandbytes \
  "httpx>=0.28.1,<1.0.0" \
  "googletrans==4.0.2"

In [2]:
import os
import time
import json
import asyncio
import nest_asyncio
from tqdm.auto import tqdm

import torch
from datasets import load_dataset
from transformers import (
    Blip2Processor,
    Blip2ForConditionalGeneration,
    BitsAndBytesConfig
)
from huggingface_hub import snapshot_download
from googletrans import Translator

from google.colab import userdata

nest_asyncio.apply()

In [3]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Hiện chưa có GPU. Đừng chạy BLIP-2 FLAN-T5-XL trên CPU.")

CUDA available: True
GPU: Tesla T4


In [4]:
try:
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        print("✅ Đã lấy HF_TOKEN từ Colab Secrets")
    else:
        print("⚠️ HF_TOKEN rỗng")
except Exception:
    hf_token = None
    print("⚠️ Không tìm thấy HF_TOKEN trong Colab Secrets")

⚠️ Không tìm thấy HF_TOKEN trong Colab Secrets


In [5]:
print("📂 Đang tải tập dữ liệu TEST...")

dataset = load_dataset(
    "pqthinh232/HCMUS-Vietnamese-Image-captioning-for-visually-impaired",
    data_files={"test": "test/**"},
    split="test",
    token=hf_token
)

print(f"✅ Thành công! Đã nạp {len(dataset)} sample.")

📂 Đang tải tập dữ liệu TEST...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/801 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/801 [00:00<?, ?it/s]

✅ Thành công! Đã nạp 4000 sample.


In [6]:
MODEL_ID = "Salesforce/blip2-flan-t5-xl"

PROMPT_TEXT_EN = (
    "Write exactly one short sentence in English (no more than 60 words) "
    "describing the main object or obstacle in the image and giving safe "
    "navigation guidance for a blind person, with no additional explanation."
)

GENERATION_CONFIG = dict(
    max_new_tokens=90,
    min_new_tokens=25,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.15,
    no_repeat_ngram_size=3,
)

OUTPUT_FILE = "results_blip2_full.json"

In [7]:
processor = Blip2Processor.from_pretrained(MODEL_ID, use_fast=False)

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=quantization_config,
)

model.eval()

print("Model loaded.")
if hasattr(model, "hf_device_map"):
    print("hf_device_map =", model.hf_device_map)

total_params = sum(p.numel() for p in model.parameters())
print(f"📦 Tổng số tham số: {total_params / 1e6:.2f} M")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded.
hf_device_map = {'': 0}
📦 Tổng số tham số: 3942.45 M


In [8]:
def get_actual_disk_size(model_id):
    model_path = snapshot_download(repo_id=model_id, local_files_only=True)
    total_size = 0

    for dirpath, _, filenames in os.walk(model_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            real_fp = os.path.realpath(fp)
            if os.path.exists(real_fp):
                total_size += os.path.getsize(real_fp)

    return total_size / (1024**3)

actual_disk_size = get_actual_disk_size(MODEL_ID)
print(f"💾 Disk Size: {actual_disk_size:.2f} GB")

💾 Disk Size: 14.69 GB


In [9]:
def translate_en_to_vi_sync(text: str, translator: Translator) -> str:
    if not text or text == "[EMPTY_OUTPUT]":
        return "[EMPTY_OUTPUT]"

    try:
        coro = translator.translate(text, src="en", dest="vi")
        loop = asyncio.get_event_loop()
        result = loop.run_until_complete(coro)

        translated_text = result.text.strip() if result and result.text else ""
        return translated_text if translated_text else "[TRANSLATION_ERROR]"

    except Exception as e:
        print(f"[WARN] Translation error: {e}")
        return "[TRANSLATION_ERROR]"


unique_images = {}
print("🔍 Đang trích xuất tên file gốc và lọc ảnh duy nhất...")

for item in dataset:
    full_path = getattr(item["image"], "filename", None)

    if full_path:
        f_name = os.path.basename(full_path)
    else:
        f_name = f"unknown_{len(unique_images)}.jpg"

    if f_name not in unique_images:
        unique_images[f_name] = item["image"]

unique_file_names = sorted(list(unique_images.keys()))
print(f"✅ Thành công! Đã tìm thấy {len(unique_file_names)} ảnh unique.")
print("Ví dụ 10 ảnh đầu:", unique_file_names[:10])

🔍 Đang trích xuất tên file gốc và lọc ảnh duy nhất...
✅ Thành công! Đã tìm thấy 800 ảnh unique.
Ví dụ 10 ảnh đầu: ['00003.jpg', '00009.jpg', '00014.jpg', '00027.jpg', '00030.jpg', '00048.jpg', '00061.jpg', '00080.jpg', '00087.jpg', '00103.jpg']


In [10]:
references_map = {}

for item in dataset:
    full_path = getattr(item["image"], "filename", None)
    if not full_path:
        continue

    f_name = os.path.basename(full_path)

    ref_text = None
    for key in ["caption", "text", "reference", "references"]:
        if key in item and item[key] is not None:
            ref_text = item[key]
            break

    if f_name not in references_map:
        references_map[f_name] = []

    if isinstance(ref_text, str):
        ref_text = ref_text.strip()
        if ref_text and ref_text not in references_map[f_name]:
            references_map[f_name].append(ref_text)

    elif isinstance(ref_text, list):
        for x in ref_text:
            if isinstance(x, str):
                x = x.strip()
                if x and x not in references_map[f_name]:
                    references_map[f_name].append(x)

print("✅ Đã tạo references_map.")

✅ Đã tạo references_map.


In [11]:
def run_debug_one(file_name):
    image = unique_images[file_name].convert("RGB")

    inputs = processor(
        images=image,
        text=PROMPT_TEXT_EN,
        return_tensors="pt"
    )

    processed_inputs = {}
    for k, v in inputs.items():
        if torch.is_tensor(v):
            processed_inputs[k] = v.to(model.device)
        else:
            processed_inputs[k] = v

    translator = Translator()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    print("🧪 DEBUG 1 ảnh:", file_name)
    print("generation_config =", GENERATION_CONFIG)

    t0 = time.time()
    with torch.no_grad():
        generated_ids = model.generate(
            **processed_inputs,
            **GENERATION_CONFIG
        )
    t1 = time.time()

    prediction_en = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0].strip()

    if prediction_en == "":
        prediction_en = "[EMPTY_OUTPUT]"

    prediction_vi = translate_en_to_vi_sync(prediction_en, translator)

    print("EN :", prediction_en)
    print("VI :", prediction_vi)
    print(f"Time: {t1 - t0:.2f}s")

    if torch.cuda.is_available():
        print(f"Peak VRAM: {torch.cuda.max_memory_allocated() / (1024**3):.2f} GB")

In [12]:
run_debug_one(unique_file_names[0])

🧪 DEBUG 1 ảnh: 00003.jpg
generation_config = {'max_new_tokens': 90, 'min_new_tokens': 25, 'do_sample': True, 'temperature': 0.7, 'top_p': 0.9, 'repetition_penalty': 1.15, 'no_repeat_ngram_size': 3}
EN : a road with trees and grasses on the right side of the image  google earth  image size 1
VI : một con đường có cây cỏ ở bên phải ảnh google Earth kích thước hình ảnh 1
Time: 12.72s
Peak VRAM: 4.16 GB


In [13]:
def run_inference(file_names_subset, output_file):
    total_time = 0.0
    results_data = []

    translator = Translator()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    print(f"🚀 Bắt đầu xử lý {len(file_names_subset)} ảnh...")
    print("generation_config =", GENERATION_CONFIG)

    for image_id, f_name in enumerate(tqdm(file_names_subset, desc="Processing Images")):
        image = unique_images[f_name].convert("RGB")

        inputs = processor(
            images=image,
            text=PROMPT_TEXT_EN,
            return_tensors="pt"
        )

        processed_inputs = {}
        for k, v in inputs.items():
            if torch.is_tensor(v):
                processed_inputs[k] = v.to(model.device)
            else:
                processed_inputs[k] = v

        t0 = time.time()
        with torch.no_grad():
            generated_ids = model.generate(
                **processed_inputs,
                **GENERATION_CONFIG
            )
        t1 = time.time()

        total_time += (t1 - t0)

        prediction_en = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )[0].strip()

        if prediction_en == "":
            prediction_en = "[EMPTY_OUTPUT]"

        prediction_vi = translate_en_to_vi_sync(prediction_en, translator)

        refs = references_map.get(f_name, [])

        results_data.append({
            "image_id": image_id,
            "prediction": prediction_vi,
            "references": refs
        })

        if image_id < 3:
            print(f"\n--- DEBUG {f_name} ---")
            print("EN :", repr(prediction_en))
            print("VI :", repr(prediction_vi))
            print(f"Time: {t1 - t0:.2f}s")

    avg_time = total_time / len(file_names_subset) if len(file_names_subset) > 0 else 0.0
    peak_vram = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0.0

    final_output = {
        "system_metrics": {
            "params_M": round(total_params / 1e6, 2),
            "disk_size_GB": round(actual_disk_size, 2),
            "time_per_img_sec": round(avg_time, 4),
            "peak_vram_GB": round(peak_vram, 2)
        },
        "predictions": results_data
    }

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(final_output, f, ensure_ascii=False, indent=4)

    print(f"✅ Đã lưu kết quả tại: {output_file}")
    return final_output

In [15]:
full_output = run_inference(
    file_names_subset=unique_file_names,
    output_file=OUTPUT_FILE
)

🚀 Bắt đầu xử lý 800 ảnh...
generation_config = {'max_new_tokens': 90, 'min_new_tokens': 25, 'do_sample': True, 'temperature': 0.7, 'top_p': 0.9, 'repetition_penalty': 1.15, 'no_repeat_ngram_size': 3}


Processing Images:   0%|          | 0/800 [00:00<?, ?it/s]


--- DEBUG 00003.jpg ---
EN : 'a road, a tree, and some grasses image of a small road through a forest in a rural area with no traffic'
VI : 'một con đường, một cái cây và một vài ngọn cỏ Hình ảnh một con đường nhỏ xuyên qua khu rừng ở một vùng nông thôn không có xe cộ qua lại'
Time: 5.73s

--- DEBUG 00009.jpg ---
EN : 'a street with shops and buildings in the background and a tai chinese style shop fronts'
VI : 'một con phố với các cửa hàng và tòa nhà ở phía sau và mặt tiền cửa hàng kiểu Thái Trung Quốc'
Time: 4.18s

--- DEBUG 00014.jpg ---
EN : 'a street view of the city with a city skyline senciing the building and street scene with traffic'
VI : 'quang cảnh đường phố của thành phố với đường chân trời của thành phố bao quanh tòa nhà và khung cảnh đường phố với giao thông'
Time: 5.03s
[WARN] Translation error: 
✅ Đã lưu kết quả tại: results_blip2_full.json


In [16]:
print(json.dumps(full_output["system_metrics"], ensure_ascii=False, indent=4))
print(json.dumps(full_output["predictions"][:3], ensure_ascii=False, indent=4))

{
    "params_M": 3942.45,
    "disk_size_GB": 14.69,
    "time_per_img_sec": 5.5797,
    "peak_vram_GB": 4.18
}
[
    {
        "image_id": 0,
        "prediction": "một con đường, một cái cây và một vài ngọn cỏ Hình ảnh một con đường nhỏ xuyên qua khu rừng ở một vùng nông thôn không có xe cộ qua lại",
        "references": [
            "con đường phía trước thẳng tắp và vắng phương tiện qua lại với biển báo giao thông nằm ở lề đất bên phải cạnh rừng thông, bạn hãy đi sát mép trái để tránh vật cản ven đường",
            "lòng đường trải nhựa rộng rãi được bao quanh bởi hàng cây thông cao vút hai bên và hiện tại không có xe cộ lưu thông, bạn có thể tự tin bước đi dọc theo sát mép đường bằng phẳng",
            "phía trước là đoạn đường trống trải có biển cảnh báo giao thông bên lề phải và cỏ mọc um tùm sát hai bên mép đường, bạn nên đi chậm trên phần đường nhựa để không vấp phải cỏ rậm",
            "không gian khu vực này rất thoáng đãng với dải phân cách làn đường rõ ràng nhưng khô

In [17]:
!pip install -q pyvi
!pip install -q git+https://github.com/salaniz/pycocoevalcap.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 69.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [18]:
import json
from pyvi import ViTokenizer
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.rouge.rouge import Rouge

# ==========================================
# 1. LOAD DỮ LIỆU TỪ FILE JSON KẾT QUẢ BLIP
# ==========================================
file_path = '/content/results_blip2_full.json'   # sửa nếu file bạn nằm chỗ khác

with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

predictions_data = data['predictions']
system_metrics = data['system_metrics']

# ==========================================
# 2. TIỀN XỬ LÝ: TÁCH TỪ TIẾNG VIỆT
# ==========================================
print("✂️ Đang thực hiện Tách từ (Word Segmentation) Tiếng Việt...")

gts = {}
res = {}

for item in predictions_data:
    img_id = str(item['image_id'])

    pred_text = item.get('prediction', '').strip().lower()
    refs_text = item.get('references', [])

    # bỏ mẫu lỗi / rỗng
    if not pred_text or len(refs_text) == 0:
        continue

    pred_seg = ViTokenizer.tokenize(pred_text)
    res[img_id] = [pred_seg]

    refs_seg = [
        ViTokenizer.tokenize(ref.strip().lower())
        for ref in refs_text
        if isinstance(ref, str) and ref.strip()
    ]

    if len(refs_seg) == 0:
        continue

    gts[img_id] = refs_seg

# đồng bộ key giữa gts và res
common_keys = sorted(list(set(gts.keys()) & set(res.keys())))
gts = {k: gts[k] for k in common_keys}
res = {k: res[k] for k in common_keys}

print(f"✅ Số ảnh hợp lệ để chấm: {len(common_keys)}")

# ==========================================
# 3. TÍNH TOÁN CÁC ĐỘ ĐO NGÔN NGỮ
# ==========================================
print("📈 Đang tính điểm BLEU, CIDEr, ROUGE-L...")

scorers = [
    (Bleu(4), ["BLEU_1", "BLEU_2", "BLEU_3", "BLEU_4"]),
    (Cider(), "CIDEr"),
    (Rouge(), "ROUGE_L")
]

final_scores = {}

for scorer, method in scorers:
    score, _ = scorer.compute_score(gts, res)

    if isinstance(method, list):
        for m, s in zip(method, score):
            final_scores[m] = round(s * 100, 2)
    else:
        final_scores[method] = round(score * 100, 2)

# ==========================================
# 4. IN BÁO CÁO TỔNG HỢP
# ==========================================
print("\n" + "★" * 60)
print("🏆 BÁO CÁO KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH BLIP2-FLAN-T5-XL 🏆")
print("★" * 60)

print("\n🅰️ [CHẤT LƯỢNG NGÔN NGỮ - TEXT METRICS]")
print(f"   ➤ BLEU-1 : {final_scores['BLEU_1']}")
print(f"   ➤ BLEU-2 : {final_scores['BLEU_2']}")
print(f"   ➤ BLEU-3 : {final_scores['BLEU_3']}")
print(f"   ➤ BLEU-4 : {final_scores['BLEU_4']}")
print(f"   ➤ CIDEr  : {final_scores['CIDEr']}")
print(f"   ➤ ROUGE-L: {final_scores['ROUGE_L']}")

print("\n🅱️ [HIỆU NĂNG HỆ THỐNG - SYSTEM METRICS]")
print(f"   ➤ Tổng tham số (Params) : {system_metrics['params_M']} M")
print(f"   ➤ Dung lượng đĩa (Disk) : {system_metrics['disk_size_GB']} GB")
print(f"   ➤ Độ trễ (Time/Img)     : {system_metrics['time_per_img_sec']} giây/ảnh")
print(f"   ➤ VRAM đỉnh (Peak VRAM) : {system_metrics['peak_vram_GB']} GB")

print("\n" + "★" * 60)

# ==========================================
# 5. CẬP NHẬT KẾT QUẢ VÀO FILE JSON
# ==========================================
data['evaluation_results'] = final_scores

with open(file_path, 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print(f"💾 Bảng điểm đã được lưu đè vào file: {file_path}")

✂️ Đang thực hiện Tách từ (Word Segmentation) Tiếng Việt...
✅ Số ảnh hợp lệ để chấm: 800
📈 Đang tính điểm BLEU, CIDEr, ROUGE-L...
{'testlen': 18967, 'reflen': 23745, 'guess': [18967, 18167, 17367, 16567], 'correct': [5818, 986, 311, 77]}
ratio: 0.7987786902505454

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🏆 BÁO CÁO KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH BLIP2-FLAN-T5-XL 🏆
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

🅰️ [CHẤT LƯỢNG NGÔN NGỮ - TEXT METRICS]
   ➤ BLEU-1 : 23.84
   ➤ BLEU-2 : 10.03
   ➤ BLEU-3 : 5.19
   ➤ BLEU-4 : 2.67
   ➤ CIDEr  : 4.52
   ➤ ROUGE-L: 13.56

🅱️ [HIỆU NĂNG HỆ THỐNG - SYSTEM METRICS]
   ➤ Tổng tham số (Params) : 3942.45 M
   ➤ Dung lượng đĩa (Disk) : 14.69 GB
   ➤ Độ trễ (Time/Img)     : 5.5797 giây/ảnh
   ➤ VRAM đỉnh (Peak VRAM) : 4.18 GB

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
💾 Bảng điểm đã được lưu đè vào file: /content/results_blip2_full.json
